# Generating test datasets

The eval workflow: **draft a prompt → build a dataset → run each case through Claude →
grade the outputs → change the prompt and compare the average score.**

This lesson builds the dataset: a JSON array of test cases, written by hand or generated by Claude.
A fast model like Haiku is fine for generating test data.

The prompt we are going to evaluate (used in the next lessons) turns an AWS task into
Python, JSON or a regex. Each test case has a `task` and the expected `format`
(the `format` field is used by the code grader in lesson 04).

## Setup

The client reads `ANTHROPIC_API_KEY` (and `ANTHROPIC_BASE_URL`, if set) from `.env` in the repo root.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Helpers

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    # The API rejects system=None, so optional params are only sent when set
    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return "".join(block.text for block in message.content if block.type == "text")

## Generate the dataset

Same trick as in "Structured data": prefill `` ```json `` and stop at `` ``` ``.

In [3]:
import json


def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "python" or "json" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

[
  {
    "task": "Parse an AWS S3 bucket name from an S3 URI in the format 's3://bucket-name/key/path'",
    "format": "regex"
  },
  {
    "task": "Create a JSON CloudFormation template snippet that defines an IAM role with a trust policy allowing EC2 service to assume the role",
    "format": "json"
  },
  {
    "task": "Write a Python function that extracts the region from an AWS ARN string (e.g., 'arn:aws:s3:us-east-1:123456789012:bucket/my-bucket')",
    "format": "python"
  }
]


## Save it

`dataset.json` is written only if it does not exist yet: to compare prompts fairly,
every version must be graded on the same cases. Delete the file to get a new dataset.

In [5]:
import os

if not os.path.exists("dataset.json"):
    with open("dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)
    print("dataset.json written")
else:
    print("dataset.json already exists, kept it")

dataset.json already exists, kept it
